<a href="https://colab.research.google.com/github/ElaineNguyen1199/Cryptography/blob/main/Week05B_Keyed_Hashing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Crypto Lab 05B — Keyed Hashing
## MACs, PRFs, HMAC, CMAC, Poly1305, and Timing-Safe Verification

**Course:** CYBR 3570 — Applied Cryptography  
**Toolkit Version:** v0.6b  

### Today's Big Question

> If anyone can compute a hash, how do we prove that a message came from someone who knows a secret key?

Last time, we studied unkeyed hash functions such as SHA-256, SHA-3, and BLAKE2. This notebook extends that discussion to **keyed hashing**.

A keyed hash can authenticate a message. That means it helps prove both that the message was not modified and that it was produced by someone who knows the shared secret key.

## Learning Objectives

By the end of this lab, you should be able to:

- explain the difference between a cryptographic hash and a MAC,
- compute and verify HMAC tags using Python,
- explain why `Hash(key || message)` is not a safe general-purpose MAC construction,
- demonstrate how replay attacks work against valid MACs,
- explain why constant-time comparison matters,
- use CMAC and Poly1305 at a high level,
- add keyed hashing utilities to your cryptographic toolkit.

## Section 1 — Setup
Run the following cell first.

In [ ]:
import hashlib
import hmac
import secrets
import time

try:
    from cryptography.hazmat.primitives.cmac import CMAC
    from cryptography.hazmat.primitives.ciphers import algorithms
    from cryptography.hazmat.primitives.poly1305 import Poly1305
    CRYPTOGRAPHY_AVAILABLE = True
except Exception as exc:
    CRYPTOGRAPHY_AVAILABLE = False
    print("cryptography package not available:", exc)

## Section 2 — Concept Check
Answer these in Markdown.

1. What security property does a MAC provide that a normal hash does not?
- Prove who created the hash. We want to know who made this.
2. Why does a MAC require a secret key?
- Because it is verfied by the reciver.
3. Why is a MAC not the same thing as encryption?
- MAC is not the same thing as encryption because you can't unhash, can't retrieve the originial message.
4. What is a replay attack?
- Resending a good message later.
5. Why is a PRF considered stronger than a MAC?
- MACs have less security requirements.

## Section 3 — A Plain Hash Is Public
Anyone can compute `SHA256(message)`. That is useful for integrity checks, but it does not prove who created the hash.

In [ ]:
def sha256_digest(message: bytes) -> str:
    """Return the SHA-256 digest of a message as a hex string."""
    return hashlib.sha256(message).hexdigest()

message = b"Transfer $100 to account 12345"
print(sha256_digest(message))

09d11180d38974a9eea9725e077f43dd4de63173bf3c33b7ba93803f24ace456


### Reflection
Suppose Alice sends Bob this message and digest:

```text
Transfer $100 to account 12345
SHA256(message)
```

Why does this not prove that Alice created the message?
- Because anyone can hash this message. Hashes are not secret nor secured.

## Section 4 — HMAC: A Standard Keyed Hash Construction
Python's `hmac` module implements HMAC. HMAC combines a secret key, a message, and a hash function.

In [ ]:
def hmac_sha256(key: bytes, message: bytes) -> bytes:
    """Compute HMAC-SHA-256(key, message)."""
    return hmac.new(key, message, hashlib.sha256).digest()

key = secrets.token_bytes(32)
message = b"Transfer $100 to account 12345"
tag = hmac_sha256(key, message)

print("key:", key.hex())
print("tag:", tag.hex())

key: 0c3a1d127c8526d181aaf1669d1403f0d3e1689a5ee66483b8cb4e0166bd0a4c
tag: 4acfbf04ebcbd0a64fbb74f8054e3166692d9d91725532ed191d93b88e448f6c


## Exercise 1 — Verify an HMAC Tag
Complete the function below.

Requirements:

- recompute the HMAC tag for the received message,
- compare the expected and received tags using `hmac.compare_digest`,
- return `True` only if the tag is valid.

In [ ]:
def verify_hmac_sha256(key: bytes, message: bytes, received_tag: bytes) -> bool:
    """Verify an HMAC-SHA-256 tag."""
    # TODO: compute the expected tag
    expected_tag = hmac_sha256(key, message)

    if hmac.compare_digest(expected_tag, received_tag):
      return True

    # TODO: use hmac.compare_digest for the comparison
    return False

valid = verify_hmac_sha256(key, message, tag)
modified = verify_hmac_sha256(key, b"Transfer $900 to account 12345", tag)

print("valid tag verifies:", valid)
print("modified message verifies:", modified)

valid tag verifies: True
modified message verifies: False


## Section 5 — Why Not Just Hash `key || message`?
A tempting but dangerous construction is:

```python
SHA256(key + message)
```

This is called a **secret-prefix MAC**. It looks plausible, but it is not a safe general-purpose design. With Merkle–Damgård hashes such as SHA-256, it can be vulnerable to length-extension attacks.

In [ ]:
def insecure_secret_prefix_mac(key: bytes, message: bytes) -> bytes:
    """Educational anti-pattern: do not use this as a MAC."""
    return hashlib.sha256(key + message).digest()

bad_tag = insecure_secret_prefix_mac(key, message)
print(bad_tag.hex())

403a26e71d6c86fc5ca402e5c209a0f28751883a118ce8433fb2eba0b073db44


### Exercise 2 — Explain the Anti-Pattern
In your own words, explain why `SHA256(key || message)` is not a good replacement for HMAC.

Your answer should mention at least one of the following: length-extension attacks, accidental design mistakes, ambiguity about key/message boundaries, or the availability of standard constructions.

- `SHA256(key || message)` is not a good replacment for HMAC because SHA-256 uses a Merkle Damgard construction which allows length extension attacks when used this way. This can also lead to accidental design mistakes, while the HMAC is a standard construction specifically designed for securely combining a secret key with a hash function.

## Section 6 — Replay Attacks
A valid MAC proves that a message/tag pair was produced by someone who knows the key. It does **not** prove that the message is fresh.

In [ ]:
shared_key = secrets.token_bytes(32)

payment = b"PAY DEREK 100"
payment_tag = hmac_sha256(shared_key, payment)

print("Original verifies:", verify_hmac_sha256(shared_key, payment, payment_tag))
print("Replay verifies:", verify_hmac_sha256(shared_key, payment, payment_tag))

Original verifies: True
Replay verifies: True


### Exercise 3 — Add a Message Number
Modify the message format so that the MAC authenticates both the message number and the message body.

Example format:

```text
000001|PAY DEREK 100
000002|PAY DEREK 100
```

Then explain why a receiver must track which message numbers have already been accepted.

In [ ]:
def encode_numbered_message(number: int, body: bytes) -> bytes:
    """Encode a message number and message body into one authenticated byte string."""
    # TODO: return number and body in a clear, unambiguous format
    str_num = f"{number:05}" + "|"
    return b"" + str_num.encode() + body

numbered = encode_numbered_message(1, b"PAY DEREK 100")
numbered_tag = hmac_sha256(shared_key, numbered)

print(numbered)
print(numbered_tag.hex())

b'00001|PAY DEREK 100'
1e973c392f71085b6da9c1c0a0f579d223367cee5fe9cc06392771d82c4d6a0a


## Section 7 — Constant-Time Comparison
A common implementation mistake is to compare tags byte by byte and return as soon as the first mismatch is found. That creates a timing leak.

In [ ]:
def insecure_compare(x: bytes, y: bytes) -> bool:
    """Educational anti-pattern: variable-time comparison."""
    if len(x) != len(y):
        return False
    for i in range(len(x)):
        if x[i] != y[i]:
            return False
    return True

correct = hmac_sha256(shared_key, b"important message")
wrong_first = bytes([correct[0] ^ 1]) + correct[1:]
wrong_last = correct[:-1] + bytes([correct[-1] ^ 1])

print(insecure_compare(correct, correct))
print(insecure_compare(correct, wrong_first))
print(insecure_compare(correct, wrong_last))

True
False
False


### Exercise 4 — Measure Timing Differences
Complete the measurement function below and compare:

- correct tag vs. correct tag,
- correct tag vs. tag wrong in the first byte,
- correct tag vs. tag wrong in the last byte,
- the same cases using `hmac.compare_digest`.

In [ ]:
def time_comparison(compare_func, a: bytes, b: bytes, trials: int = 100_000) -> float:
    """Return elapsed time for repeated comparison calls."""
    start = time.perf_counter()
    for _ in range(trials):
        compare_func(a, b)
    end = time.perf_counter()
    return end - start

# TODO: run timing experiments with insecure_compare and hmac.compare_digest
# Example:
print(time_comparison(insecure_compare, correct, correct))
print(time_comparison(insecure_compare, correct, wrong_first))
print(time_comparison(insecure_compare, correct, wrong_last))

0.21524369600001592
0.03703958500000226
0.2245304580001175


## Section 8 — CMAC: A Block-Cipher-Based MAC
HMAC builds a MAC from a hash function. CMAC builds a MAC from a block cipher such as AES.

In [ ]:
if CRYPTOGRAPHY_AVAILABLE:
    cmac_key = secrets.token_bytes(16)  # AES-128 key
    c = CMAC(algorithms.AES(cmac_key))
    c.update(b"message authenticated with AES-CMAC")
    cmac_tag = c.finalize()
    print("CMAC tag:", cmac_tag.hex())
else:
    print("Install cryptography to run the CMAC example.")

CMAC tag: 718b337337ae1748316ca8fb46960bdb


### Exercise 5 — Compare HMAC and CMAC
Answer in Markdown:

1. What primitive does HMAC build from?
- HMAC builds a MAC from a cryptographic hash function, such as SHA-256
2. What primitive does CMAC build from?
- CMAC builds a MAC from a block cipher, such as AES
3. Why might a protocol choose CMAC instead of HMAC?
- A protocol might choose CMAC if it already uses a block cipher like AES and wants to use that existing primitive for authentication instead of adding a separate hash based MAC.
4. Why should you avoid implementing CBC-MAC yourself?
- CBC-MAC has security pitfalls, especially if it is used incorrectly or with variable-length messages. It is safer to use a standardized construction such as CMAC rather than trying to implement CBC-MAC yourself.

## Section 9 — Poly1305: A Fast Dedicated MAC
Poly1305 is a fast MAC often paired with ChaCha20 in modern authenticated encryption.

In [ ]:
if CRYPTOGRAPHY_AVAILABLE:
    poly_key = secrets.token_bytes(32)
    poly_tag = Poly1305.generate_tag(poly_key, b"message authenticated with Poly1305")
    print("Poly1305 tag:", poly_tag.hex())
else:
    print("Install cryptography to run the Poly1305 example.")

Poly1305 tag: 933c6a033feb3881b1b67650073e22f5


## Section 10 — Toolkit Integration
Add a new module to your toolkit:

```text
crypto_toolkit/
    hashes/
        keyed.py
```

Your module should include:

- `hmac_sha256(key: bytes, message: bytes) -> bytes`
- `verify_hmac_sha256(key: bytes, message: bytes, tag: bytes) -> bool`
- `sha256_digest(message: bytes) -> bytes`
- clear docstrings warning against naive keyed-hash constructions

Optional challenge: add `cmac_aes()` and `poly1305_tag()`.

In [ ]:
# Starter code for crypto_toolkit/hashes/keyed.py

MODULE_STARTER = """
"""
Module: crypto_toolkit.hashes.keyed

Educational keyed hashing utilities for CYBR 3570.

WARNING
-------
These wrappers are for learning. Production applications should rely on
well-reviewed libraries and protocol-level constructions.
"""

import hashlib
import hmac


def sha256_digest(message: bytes) -> bytes:
    """Return SHA-256(message)."""
    return hashlib.sha256(message).digest()


def hmac_sha256(key: bytes, message: bytes) -> bytes:
    """Return HMAC-SHA-256(key, message)."""
    return hmac.new(key, message, hashlib.sha256).digest()


def verify_hmac_sha256(key: bytes, message: bytes, tag: bytes) -> bool:
    """Verify HMAC-SHA-256 using constant-time comparison."""
    expected = hmac_sha256(key, message)
    return hmac.compare_digest(expected, tag)
"""

print(MODULE_STARTER)

SyntaxError: invalid syntax (3416369380.py, line 7)

## Section 11 — Security Engineering Reflection
Answer these in Markdown.

1. Why is encryption without authentication dangerous?
- Encryption protects the confidentiality of a message, but it does not necessarily stop an attacker from changing the encrypted data. Without authentication, the receiver may not know if the message was modified.
2. Why does a valid MAC not necessarily mean a message is fresh?
- A MAC only provides that the message and tag were created by someone with the secret key. An attacker could capture a valid message and MAC and replay them later. The protocol needs soemthing like a nonce, timestamp, or message number to prevent replay attacks.
3. Why is constant-time comparison part of cryptographic engineering rather than just performance engineering?
- Normal comparisons can potentially reveal information about how many characters or bytes were correct based on how long the comparison takes. Constant-time comparison helps prevent timing attacks that could leak part of the secret MAC.
4. A developer says, “I used SHA-256, so my MAC is secure.” What questions would you ask?
- I would have ask them how they are using SHA-256 with the secret key. Like for an example are they using SHA256(key || message) or just a standard construction like HMAC-SHA-256. I would also ask them how the MAC is verified and whether they use constant time comparison. I would not assume that just using SHA-256 by itself would make MAC secured.
5. What is the connection between this week's material and authenticated encryption?
- This weeks material explains how authentication can protect the integrity and authenticity of data, while encryption protects confidentiality. Authenticated encryption combines both protections so that a message is kept secret and the reciever can also detect if it was modified in any way.

## Submission Checklist

- [ ] I answered all concept questions.
- [ ] I completed `verify_hmac_sha256()`.
- [ ] I completed the replay/message-number exercise.
- [ ] I ran the timing comparison experiment.
- [ ] I added `crypto_toolkit/hashes/keyed.py` to my toolkit.
- [ ] I committed and pushed my changes.